<!DOCTYPE html>
<html>
<head>
  <meta charset="UTF-8">
  <style>
    #vis.vega-embed {
      width: 100%;
      display: flex;
    }

    #vis.vega-embed details,
    #vis.vega-embed details summary {
      position: relative;
    }
  </style>
  <script type="text/javascript" src="https://cdn.jsdelivr.net/npm/vega@6"></script>
  <script type="text/javascript" src="https://cdn.jsdelivr.net/npm/vega-lite@6.1.0"></script>
  <script type="text/javascript" src="https://cdn.jsdelivr.net/npm/vega-embed@7"></script>
</head>
<body>
  <div id="vis"></div>
  <script>
    (function(vegaEmbed) {
      var spec = {"config": {"view": {"continuousWidth": 300, "continuousHeight": 300}}, "data": {"name": "data-0c84ee3d6606e082f376fcaa2986c275"}, "mark": {"type": "bar", "color": "#e0754a"}, "encoding": {"tooltip": [{"field": "borough", "title": "Borough", "type": "nominal"}, {"field": "avg_price", "format": "$.2f", "title": "Average Price", "type": "quantitative"}], "x": {"field": "borough", "sort": ["Bronx", "Brooklyn", "Manhattan", "Queens", "Staten Island"], "title": "NYC Borough", "type": "nominal"}, "y": {"field": "avg_price", "title": "Average Price (USD)", "type": "quantitative"}}, "height": 350, "params": [{"name": "room_type", "bind": {"input": "select", "options": ["All", "Entire home/apt", "Private room", "Shared room"], "name": "Room Type: "}, "value": "All"}], "title": "Average Nightly Price by Borough", "transform": [{"filter": "(datum.room_type === room_type)"}], "width": 500, "$schema": "https://vega.github.io/schema/vega-lite/v6.1.0.json", "datasets": {"data-0c84ee3d6606e082f376fcaa2986c275": [{"borough": "Bronx", "room_type": "Entire home/apt", "avg_price": 118.78947368421052}, {"borough": "Bronx", "room_type": "Private room", "avg_price": 56.44642857142857}, {"borough": "Bronx", "room_type": "Shared room", "avg_price": 44.0}, {"borough": "Brooklyn", "room_type": "Entire home/apt", "avg_price": 156.2117288651942}, {"borough": "Brooklyn", "room_type": "Private room", "avg_price": 80.51359516616314}, {"borough": "Brooklyn", "room_type": "Shared room", "avg_price": 68.45833333333333}, {"borough": "Manhattan", "room_type": "Entire home/apt", "avg_price": 182.18850574712644}, {"borough": "Manhattan", "room_type": "Private room", "avg_price": 101.42755344418052}, {"borough": "Manhattan", "room_type": "Shared room", "avg_price": 87.4888888888889}, {"borough": "Queens", "room_type": "Entire home/apt", "avg_price": 128.0632911392405}, {"borough": "Queens", "room_type": "Private room", "avg_price": 72.7286432160804}, {"borough": "Queens", "room_type": "Shared room", "avg_price": 87.5}, {"borough": "Staten Island", "room_type": "Entire home/apt", "avg_price": 154.46153846153845}, {"borough": "Staten Island", "room_type": "Private room", "avg_price": 61.833333333333336}, {"borough": "Bronx", "room_type": "All", "avg_price": 71.15384615384616}, {"borough": "Brooklyn", "room_type": "All", "avg_price": 123.04678111587982}, {"borough": "Manhattan", "room_type": "All", "avg_price": 149.22217153284672}, {"borough": "Queens", "room_type": "All", "avg_price": 96.90243902439025}, {"borough": "Staten Island", "room_type": "All", "avg_price": 100.6774193548387}]}};
      var embedOpt = {"mode": "vega-lite"};

      function showError(el, error){
          el.innerHTML = ('<div style="color:red;">'
                          + '<p>JavaScript Error: ' + error.message + '</p>'
                          + "<p>This usually means there's a typo in your chart specification. "
                          + "See the javascript console for the full traceback.</p>"
                          + '</div>');
          throw error;
      }
      const el = document.getElementById('vis');
      vegaEmbed("#vis", spec, embedOpt)
        .catch(error => showError(el, error));
    })(vegaEmbed);

  </script>
</body>
</html>

# Airbnb NYC Visualization 5

This notebook builds an interactive geographic map of Airbnb listing prices across the NYC boroughs, colored by price.


In [ ]:
# used https://plotly.com/python/scatter-plots-on-maps/ for information on graphing with plotly
# learned to make dropdown menu in ploty at https://plotly.com/python/dropdowns/
import pandas as pd
import plotly.graph_objects as go


CSV_PATH = "AB_NYC_2019_clean_5000.csv"
OUTPUT_PATH = "viz5_map.html"

PRICE_BINS = [0, 50, 100, 150, 250, float("inf")]
PRICE_LABELS = ["$0-50", "$51-100", "$101-150", "$151-250", "$250+"]
BOROUGH_ORDER = ["All", "Bronx", "Brooklyn", "Manhattan", "Queens", "Staten Island"]

df = pd.read_csv(CSV_PATH)
df["price_cat"] = pd.cut(
    df["price"],
    bins = PRICE_BINS,
    labels = PRICE_LABELS,
    right = True,
).astype(str)

fig = go.Figure()

for borough in BOROUGH_ORDER:
    subset = df if borough == "All" else df[df["neighbourhood_group"] == borough]
    fig.add_trace(go.Scattergeo(
        lon = subset["longitude"],
        lat = subset["latitude"],
        text = subset["name"] + "<br>" +
        "Borough: " + subset["neighbourhood_group"] + "<br>" + 
        "Neighborhood: " + subset["neighbourhood"] + "<br>" + 
        "Room Type: " + subset["room_type"] + "<br>" + 
        "Price: $" + subset["price"].astype(str),
        mode = "markers",
        marker = dict(
            size = 4,
            opacity = 0.6,
            color = subset["price"],
            colorscale = "YlOrRd",  # found colormap at https://plotly.com/python/builtin-colorscales/
            reversescale = False,
            colorbar = dict(title = dict(text = "Price (USD)")),
            cmin = 0,
            cmax = 300,
        ),
        name = borough,
        visible = (borough == "All"),
))

fig.update_layout(
    title = dict(text = "NYC Airbnb Listings by Price", y = 0.97),
    updatemenus = [
        dict(
            active = 0,
            buttons = [
                  dict(
                      label = borough,
                      method = "update",
                      args = [{"visible": [b == borough for b in BOROUGH_ORDER]}, 
                              {"title": f"NYC Airbnb Listings by Price - {borough}"}]
                )
                for borough in BOROUGH_ORDER
            ],
            direction = "down",
            showactive = True,
            x = 0.01,
            xanchor = "left",
            y = 0.92,
            yanchor = "top",
        )],
geo = dict(
        scope = "usa",
        showland = True,
        landcolor = "rgb(212, 212, 212)",
        showlakes = True,
        lakecolor = "rgb(255, 255, 255)",
        resolution = 50,
        lonaxis = dict(range = [-74.26, -73.68]),
        lataxis = dict(range = [40.49, 40.93]),
        fitbounds = "locations"
    ),
    height = 700,
    margin = dict(t=80, b=0, l=0, r=0)
)

fig.write_html(OUTPUT_PATH)
print(f"Saved -> {OUTPUT_PATH}")
fig.show()


Saved -> viz5_map.html
